# Interest-Rate Shock Sensitivity: Research Design and Data Audit

This notebook begins a new extension of the asset-embedding project. It asks whether the geometry learned from institutional holdings encodes economically familiar exposure to unexpected monetary-policy news.

The notebook is deliberately an audit and design gate. It freezes the timing rules, inventories the official shock data and all reusable local inputs, and estimates the new CRSP download. It does **not** fit a model or launch a large WRDS query.

## 1. Setup

All downloaded and generated inputs are checkpointed locally. The setup locates the repository whether Jupyter starts in the repository root or in this notebook's directory.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display

def locate_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'full_replication_80q').exists() and (candidate / 'remaining_benchmarks').exists():
            return candidate
    raise FileNotFoundError('Could not locate the Asset_Embeddings repository root.')

ROOT = locate_root()
MODULE = ROOT / 'economic_shock_sensitivity'
DATA = MODULE / 'data'
RAW = DATA / 'raw' / 'sf_fed'
RESULTS = MODULE / 'results' / 'design_audit'
MANIFESTS = MODULE / 'manifests'
for path in [RAW, RESULTS, MANIFESTS]: path.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(MODULE / 'src'))
from rate_shock_audit import (download_checkpoint, inventory_excel, inventory_zip,
    inspect_tabular_file, parquet_audit, quarter_labels, save_json)
print('Repository root:', ROOT)

## 2. Freeze the empirical contract

The primary target is a continuous firm-level sensitivity to high-frequency FOMC surprises. A firm's target at an anchor quarter will be estimated only from earlier events. The corresponding embedding must come from the preceding quarter, which is a conservative guard against the reporting delay in institutional holdings.

Raw embedding coordinates are not pooled across quarters because independently fitted SVD spaces may rotate or reflect. Models will initially be fitted cross-sectionally within each anchor quarter, with a permanent firm split used for honest aggregation across time.

In [ ]:
DESIGN = {
    'version': 'rate-shock-v1',
    'question': 'Do holdings embeddings encode firm sensitivity to unexpected monetary-policy shocks?',
    'primary_shock_source': 'Federal Reserve Bank of San Francisco USMPD',
    'primary_shock_choice': 'to_be_frozen_after_column_audit',
    'shock_convention': 'positive must mean monetary tightening; reverse source sign if necessary',
    'firm_return': 'CRSP daily PERMCO value-weighted return less CRSP value-weighted market return',
    'primary_event_window': '[0,0] FOMC communication day',
    'robustness_event_window': '[0,+1] trading days',
    'historical_beta_window_years': 5,
    'minimum_prior_events': 20,
    'target_controls': ['market beta robustness', 'winsorization', 'empirical-Bayes reliability shrinkage'],
    'embedding_information_lag': 'previous calendar-quarter embedding',
    'coordinate_policy': 'quarter-specific regressions; never pool unaligned raw coordinates',
    'firm_split': {'method': 'permanent SHA-256 split on PERMCO', 'train': .60, 'validation': .20, 'test': .20},
    'time_split': {'development': 'through 2016', 'validation': '2017-2020', 'final_test': '2021-2024'},
    'primary_embeddings': ['all_s12_rs_binary_10d', 'all_s12_rs_l_min_10d'],
    'behavioral_embeddings': ['stable_stock_picker', 'tactical_active', 'benchmark_rebalancer', 'passive_like'],
    'benchmarks': ['characteristics', 'industry', 'characteristics_plus_embedding', 'shuffled_embedding', 'training_mean'],
    'confirmatory_metrics': ['out_of_sample_R2_for_historical_beta', 'future_event_interaction_R2', 'future_event_rank_correlation'],
    'no_text_or_llm_dependency': True,
}
design_path = MANIFESTS / 'rate_shock_design_v1.json'
save_json(DESIGN, design_path)
display(pd.json_normalize(DESIGN, sep='.').T.rename(columns={0: 'frozen_value'}))
print('Saved:', design_path)

## 3. Download the official shock files

The San Francisco Fed's U.S. Monetary Policy Event-Study Database contains high-frequency changes in rates and asset prices around FOMC statements, press conferences, combined monetary events, and minutes. The companion archive supplies monetary-policy-surprise constructions. Both small official files are saved byte-for-byte with hashes so later notebooks use a frozen source.

Sources: [USMPD landing page](https://www.frbsf.org/research-and-insights/data-and-indicators/us-monetary-policy-event-study-database/) and [SF Fed monetary-policy surprises](https://www.frbsf.org/research-and-insights/data-and-indicators/monetary-policy-surprises/).

In [ ]:
SOURCE_URLS = {
    'USMPD.xlsx': 'https://www.frbsf.org/wp-content/uploads/USMPD.xlsx',
    'monetary-policy-surprises.zip': 'https://www.frbsf.org/wp-content/uploads/monetary-policy-surprises.zip',
}
download_audit = pd.DataFrame([download_checkpoint(url, RAW / name) for name, url in SOURCE_URLS.items()])
download_audit.to_parquet(RESULTS / 'official_source_download_audit.parquet', index=False)
display(download_audit)

## 4. Inventory the official files before choosing a shock

We do not select a convenient-looking column silently. This cell exposes every workbook sheet and archive member, its dimensions, date candidates, and numeric fields. The next iteration will freeze one primary tightening-surprise series and document its sign, units, event type, and coverage.

In [ ]:
excel_inventory, usmpd_sheets = inventory_excel(RAW / 'USMPD.xlsx')
zip_inventory = inventory_zip(RAW / 'monetary-policy-surprises.zip', RAW / 'monetary_policy_surprises')
tabular_inventory = pd.DataFrame([inspect_tabular_file(Path(path)) for path in zip_inventory.local_path])
excel_inventory.to_parquet(RESULTS / 'usmpd_excel_inventory.parquet', index=False)
zip_inventory.to_parquet(RESULTS / 'surprise_archive_inventory.parquet', index=False)
tabular_inventory.to_parquet(RESULTS / 'surprise_tabular_inventory.parquet', index=False)
print('USMPD workbook sheets')
display(excel_inventory)
print('Monetary-surprise archive members')
display(zip_inventory)
display(tabular_inventory)

## 5. Audit the 80-quarter all-S12 embedding backbone

The rate experiment reuses the established asset embeddings. We require both RS-Binary and RS-L-Min for every quarter from 2005Q1 through 2024Q4, a unique PERMCO key, and at least ten dimensions. This is a read-only audit.

In [ ]:
expected_quarters = quarter_labels()
embedding_records = []
for label in expected_quarters:
    for model in ['rs_binary', 'rs_l_min']:
        path = ROOT / 'full_replication_80q' / 'embeddings' / label / model / 'asset_embeddings_normalized_labeled.parquet'
        record = parquet_audit(path, f'{label}/{model}')
        record.update(quarter=label, model=model)
        embedding_records.append(record)
embedding_audit = pd.DataFrame(embedding_records)
embedding_audit.to_parquet(RESULTS / 'all_s12_embedding_audit.parquet', index=False)
display(embedding_audit.groupby('model', as_index=False).agg(quarters=('quarter','nunique'), existing=('exists','sum'), minimum_firms=('rows','min'), maximum_firms=('rows','max'), minimum_dimensions=('embedding_dimensions','min'), duplicate_keys=('duplicate_keys','sum')))
assert embedding_audit.exists.all() and embedding_audit.embedding_dimensions.ge(10).all()


## 6. Audit behavioral embedding variants and characteristics

The extension will compare embeddings learned from different investor behaviors, but the all-S12 embeddings remain the primary specification. Characteristics and industry controls are benchmarks rather than ingredients used to construct the shock target.

In [ ]:
styles = ['stable_stock_picker','tactical_active','benchmark_rebalancer','passive_like']
behavior_records, characteristic_records = [], []
for label in expected_quarters:
    characteristic_path = ROOT / 'remaining_benchmarks' / 'return_comovement' / 'characteristics' / f'{label}.parquet'
    characteristic_records.append(parquet_audit(characteristic_path, label))
    for style in styles:
        for model in ['rs_binary','rs_l_min']:
            path = ROOT / 'active_passive_extension' / 'benchmarks' / 'behavioral_embeddings' / label / style / model / 'asset_embeddings_10d.parquet'
            record = parquet_audit(path, f'{label}/{style}/{model}')
            record.update(quarter=label, style=style, model=model)
            behavior_records.append(record)
behavior_audit = pd.DataFrame(behavior_records)
characteristic_audit = pd.DataFrame(characteristic_records)
behavior_audit.to_parquet(RESULTS / 'behavioral_embedding_audit.parquet', index=False)
characteristic_audit.to_parquet(RESULTS / 'characteristic_panel_audit.parquet', index=False)
display(behavior_audit.groupby(['style','model'], as_index=False).agg(expected=('quarter','size'), existing=('exists','sum'), minimum_firms=('rows','min'), maximum_firms=('rows','max')))
display(pd.DataFrame({'input':['quarterly characteristics'], 'expected':[80], 'existing':[int(characteristic_audit.exists.sum())], 'minimum_firms':[characteristic_audit.rows.min()], 'maximum_firms':[characteristic_audit.rows.max()]}))

## 7. Audit existing daily-return checkpoints

The geopolitical extension already contains CRSP daily returns for 2021–2022. They are useful for schema validation and may cover some final-test events, but they do not provide the long pre-event history needed to estimate rolling rate sensitivities.

In [ ]:
daily_path = ROOT / 'geopolitical_exposure' / 'event_data' / 'crsp_daily_security_returns_2021_2022.parquet'
if daily_path.exists():
    daily = pd.read_parquet(daily_path)
    daily_summary = pd.DataFrame([{
        'exists': True, 'rows': len(daily), 'permnos': daily.permno.nunique(), 'permcos': daily.permco.nunique(),
        'first_date': pd.to_datetime(daily.date).min(), 'last_date': pd.to_datetime(daily.date).max(),
        'stored_mb': daily_path.stat().st_size / 2**20, 'columns': ', '.join(daily.columns)
    }])
else:
    daily_summary = pd.DataFrame([{'exists': False}])
daily_summary.to_parquet(RESULTS / 'existing_daily_return_audit.parquet', index=False)
display(daily_summary)

## 8. Timing map and proposed analysis anchors

A five-year rolling target cannot be estimated reliably at the beginning of 2005 without downloading earlier CRSP history. The economical default is therefore to query event windows from 2005 onward and begin quarterly prediction anchors in 2010. This still provides 60 anchor quarters and preserves 2021–2024 as a final untouched period. Each event uses the most recently available **previous-quarter** embedding.

In [ ]:
anchors = pd.DataFrame({'target_quarter': pd.period_range('2010Q1','2024Q4',freq='Q')})
anchors['embedding_quarter'] = anchors.target_quarter - 1
anchors['time_sample'] = np.select(
    [anchors.target_quarter.dt.year.le(2016), anchors.target_quarter.dt.year.le(2020)],
    ['development','validation'], default='final_test')
anchors['target_quarter'] = anchors.target_quarter.astype(str).str.lower()
anchors['embedding_quarter'] = anchors.embedding_quarter.astype(str).str.lower()
anchors.to_parquet(RESULTS / 'quarter_timing_map.parquet', index=False)
display(anchors.groupby('time_sample', as_index=False).agg(first_target=('target_quarter','min'), last_target=('target_quarter','max'), quarters=('target_quarter','size')))
display(anchors.head())

## 9. Acquisition decision gate

The next notebook will first freeze the exact surprise column after inspecting the tables above. It will then construct the FOMC event-date list and download only narrow CRSP windows around those dates—not a full 20-year daily panel. Expected scale is roughly 160 meeting events from 2005–2024, two event days per robustness window, and the CRSP common-stock universe on those dates.

No WRDS query is run here. That separation makes the expensive acquisition deliberate, restartable, and locally checkpointed.

In [ ]:
decision = pd.DataFrame([
    {'component':'Official high-frequency shock source', 'status':'downloaded_and_inventoried', 'next_action':'freeze column, sign, units, and FOMC event filter'},
    {'component':'All-S12 embeddings', 'status':f'{int(embedding_audit.exists.sum())}/160 files present', 'next_action':'reuse; take first 10 dimensions per quarter'},
    {'component':'Behavioral embeddings', 'status':f'{int(behavior_audit.exists.sum())}/{len(behavior_audit)} files present', 'next_action':'reuse as secondary comparisons'},
    {'component':'Characteristics', 'status':f'{int(characteristic_audit.exists.sum())}/80 quarters present', 'next_action':'reuse as benchmarks'},
    {'component':'Daily CRSP event returns', 'status':'partial 2021-2022 checkpoint only', 'next_action':'new narrow-window WRDS extraction, approximately 2005-2024'},
    {'component':'Model estimation', 'status':'not started by design', 'next_action':'begin only after target and timing audits pass'},
])
decision.to_parquet(RESULTS / 'design_gate.parquet', index=False)
display(decision)
print('Audit artifacts:', RESULTS)
print('STOP: inspect this notebook before constructing the WRDS acquisition notebook.')